In [ ]:
import pandas as pd

# Lê o CSV com separador ';'
df = pd.read_csv("marketingdigital.csv", sep=';')

print("✅ Dados lidos corretamente:")
print(df.head())

In [ ]:
# Teste de conexão com o Azure ML
from azure.ai.ml import MLClient
from azure.identity import ClientSecretCredential

tenant_id = "xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"
client_id = "xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"
client_secret = "xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"
subscription_id = "xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"
resource_group = "ml-treinamento"
workspace = "marketing"

credential = ClientSecretCredential(
    tenant_id=tenant_id,
    client_id=client_id,
    client_secret=client_secret
)

ml_client = MLClient(
    credential=credential,
    subscription_id=subscription_id,
    resource_group_name=resource_group,
    workspace_name=workspace
)

# Validação
ws = ml_client.workspaces.get("marketing")
print(f"✅ Conexão ativa com Workspace: {ws.name} | Localização: {ws.location}")
print(f"📦 Resource Group: {ml_client.resource_group_name}")
print(f"🔑 Assinatura: {ml_client.subscription_id}")




In [ ]:
# Criação da Estrutura de Diretórios
import os

train_src_dir = "./src"
os.makedirs(train_src_dir, exist_ok=True)

In [ ]:
# Script Principal de Treinamento
%%writefile ./src/main.py
import argparse
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data", type=str, help="URL ou caminho do CSV")
    parser.add_argument("--test_train_ratio", type=float, default=0.25)
    parser.add_argument("--n_estimators", type=int, default=100)
    parser.add_argument("--learning_rate", type=float, default=0.1)
    parser.add_argument("--registered_model_name", type=str, required=True)
    args = parser.parse_args()

    mlflow.start_run()
    mlflow.sklearn.autolog()

    # Lendo CSV com separador correto
    df = pd.read_csv(args.data, sep=';')

    # Separar target e features
    target_column = "Receita Total"
    y = df.pop(target_column).astype(float)  # target como float
    X = df.astype(float).values               # features como float

    # Divisão treino/teste
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=args.test_train_ratio, random_state=42
    )

    # Treinar modelo
    reg = GradientBoostingRegressor(
        n_estimators=args.n_estimators,
        learning_rate=args.learning_rate,
        random_state=42
    )
    reg.fit(X_train, y_train)

    # Previsão e métricas
    y_pred = reg.predict(X_test)
    print("Mean Squared Error:", mean_squared_error(y_test, y_pred))
    print("R2 Score:", r2_score(y_test, y_pred))

    # Registrar modelo no MLflow (corrigido)
    mlflow.sklearn.log_model(
        sk_model=reg,
        artifact_path=args.registered_model_name,       # obrigatório
        registered_model_name=args.registered_model_name
    )

    mlflow.end_run()

if __name__ == "__main__":
    main()

In [ ]:
# Definição do Job de Treinamento
from azure.ai.ml import command, Input


registered_model_name = "modelo_regressao_marketing"

job = command(
    inputs={
        "data": Input(
            type="uri_file",
            path="https://raw.githubusercontent.com/Dayanebiaerafa/projeto-Azureml-marketing/main/marketingdigital.csv"
        ),
        "test_train_ratio": 0.25,
        "n_estimators": 100,
        "learning_rate": 0.1,
        "registered_model_name": registered_model_name,
    },
    code="./src",
    command=(
        "python main.py "
        "--data ${{inputs.data}} "
        "--test_train_ratio ${{inputs.test_train_ratio}} "
        "--learning_rate ${{inputs.learning_rate}} "
        "--n_estimators ${{inputs.n_estimators}} "
        "--registered_model_name ${{inputs.registered_model_name}}"
    ),
    environment="azureml://registries/azureml/environments/sklearn-1.5/labels/latest",
    display_name="marketing_regression_tutorial",
)

In [ ]:
# Limpeza de Endpoints com Falha
print("Endpoints existentes:")
for ep in ml_client.online_endpoints.list():
    print(ep.name, ep.provisioning_state)
    # Deletar endpoints que falharam
    if ep.provisioning_state == "Failed":
        print(f"Deletando endpoint com falha: {ep.name}")
        ml_client.online_endpoints.begin_delete(ep.name).result()

In [ ]:
# Verificação de Endpoints Existentes
for e in ml_client.online_endpoints.list():
    print(f"{e.name} | {e.provisioning_state}")

In [ ]:
# Envio do Job de Treinamento
ml_client.jobs.create_or_update(job)

In [ ]:
import uuid

# Gerar nome único para o endpoint
online_endpoint_name = f"marketing-endpoint-{uuid.uuid4().hex[:5]}"
print(f"🚀 Criando novo endpoint: {online_endpoint_name}")

In [ ]:
# Checar workspace
ws = ml_client.workspaces.get("marketing")
print(f"Workspace: {ws.name}, Localização: {ws.location}")

# Listar compute targets disponíveis
computes = ml_client.compute.list()
for c in computes:
    print(f"Compute: {c.name}, Tipo: {c.type}, Estado: {c.provisioning_state}")

In [ ]:
# Definição do Nome do Endpoint
online_endpoint_name = "marketing-endpoint-045be"

In [ ]:
# Criação do Endpoint de Inferência
from azure.ai.ml.entities import ManagedOnlineEndpoint

endpoint = ManagedOnlineEndpoint(
    name=online_endpoint_name,
    description="Endpoint de previsão marketing",
    auth_mode="key"
)

endpoint = ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print(f"✅ Endpoint criado: {endpoint.name} | Status: {endpoint.provisioning_state}")


In [ ]:
# Vamos escolher a versão mais recente do modelo
latest_model_version = max(
    [int(m.version) for m in ml_client.models.list(name=registered_model_name)]
)
print(f'Latest model is version "{latest_model_version}" ')

In [ ]:
# Recuperação do Modelo Registrado
model = ml_client.models.get(name=registered_model_name, version=latest_model_version)

In [ ]:
# Importa bibliotecas necessárias
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential

# Tenta autenticação automática; se falhar, abre login no navegador
try:
    credential = DefaultAzureCredential()
    ml_client = MLClient.from_config(credential=credential)
except Exception:
    credential = InteractiveBrowserCredential()
    ml_client = MLClient(
        credential=credential,
        subscription_id="e7f359a9-446e-46c1-a975-5ff92439c172",
        resource_group_name="ml-treinamento",
        workspace_name="marketing"
    )

print("✅ Conectado ao workspace Azure ML com sucesso!")

In [ ]:
# Recupera o modelo salvo corretamente
model = ml_client.models.get(name="modelo_regressao_marketing", version="15"

In [ ]:
# Criação do Ambiente de Execução
from azure.ai.ml.entities import Environment

env = Environment(
    name="sklearn-env-py310",
    description="Ambiente compatível com modelo_regressao_marketing (py310, sklearn 1.5.2)",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04",
    conda_file={
        "name": "sklearn-env-py310",
        "dependencies": [
            "python=3.10",
            "scikit-learn=1.5.2",
            "cloudpickle=3.0.0",
            {"pip": ["mlflow==2.13.1", "azureml-defaults"]}
        ]
    }
)

ml_client.environments.create_or_update(env)
print("✅ Ambiente criado/atualizado.")

In [ ]:
from azure.ai.ml.entities import ManagedOnlineDeployment, Environment

# Nome do endpoint existente
online_endpoint_name = "marketing-endpoint-045be"


blue = ManagedOnlineDeployment(
    name="blue",
    endpoint_name="marketing-endpoint-045be",
    model=model,
    environment=env,
    instance_type="Standard_E8ds_v4",   
    instance_count=2,                    
    autoscale_min_replicas=1,
    autoscale_max_replicas=3
)

# Recria o deployment
ml_client.begin_create_or_update(blue_deployment).result()
print("✅ Deployment recriado com modelo e ambiente explícitos!")


In [ ]:
# Preparação da Pasta de Deployd
eploy_dir = "./deploy"
os.makedirs(deploy_dir, exist_ok=True)

In [ ]:
# Criação do Arquivo de Requisição de Teste
%%writefile {deploy_dir}/sample-request.json

{
  "input_data": {
    "columns": ["Gasto Anuncios"],
    "index": [0],
    "data": [
      [382]
    ]
  }
}

In [ ]:
# Teste do Endpoint com o Modelo Implantado
response = ml_client.online_endpoints.invoke(
    endpoint_name="marketing-endpoint-045be",
    request_file="./deploy/sample-request.json",
    deployment_name="blue",
)
print(response)